# Agentic DocQA: GPU evaluation v2
Runs the actual agent with Nomic task prefixes, a six-condition retrieval ablation, deterministic diagnostic metrics and two separately prompted judge passes per answer.

Choose **Runtime > Change runtime type > T4 GPU**. Load the updated **agentic-docqa-v2.zip** from your shared Drive folder, upload it, or use GitHub with a repository-scoped Colab Secret. This notebook has no recorded benchmark outputs.

New results go to **My Drive/Agentic DocQA/evaluation-results/v2/**. Existing results remain separate. Choose **Full evaluation** to rerun the pipeline and prefix comparison, or **Rejudge saved answers** to judge all 20 old answers uniformly without retrieval/generation. The latter cannot measure the prefix fix on old vectors.


## 1. Check the GPU
Stop early if no NVIDIA GPU is assigned. Colab GPU availability and runtime length are not guaranteed.


In [ ]:
import subprocess, sys
check = subprocess.run(["bash", "-lc", "nvidia-smi"], text=True, capture_output=True)
print(check.stdout or check.stderr)
print("Python:", sys.version)
if check.returncode:
    raise RuntimeError("Choose Runtime > Change runtime type > T4 GPU, then rerun this notebook.")


## 2. Load the private project
**Drive ZIP** loads the project already saved in your Drive. **Upload ZIP** lets you select a downloaded project ZIP. Both avoid GitHub authentication. For **GitHub**, optionally add a repository-scoped token as the Colab Secret `GITHUB_TOKEN` and allow this notebook to read it. Tokens are never put in URLs, shell commands, outputs, or saved files.

Use the updated ZIP containing `eval/run_eval.py`; the older project-only ZIP cannot run the evaluation.


In [ ]:
import hashlib, io, os, shutil, stat, urllib.request, zipfile
from pathlib import Path, PurePosixPath
SOURCE = "Drive ZIP" #@param ["Drive ZIP", "Upload ZIP", "GitHub"]
DRIVE_ZIP = "/content/drive/MyDrive/Agentic DocQA/agentic-docqa-v2.zip" #@param {type:"string"}
if SOURCE == "Drive ZIP":
    from google.colab import drive
    drive.mount("/content/drive")
    archive_bytes = Path(DRIVE_ZIP).read_bytes()
elif SOURCE == "Upload ZIP":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select exactly one updated project ZIP.")
    archive_bytes = next(iter(uploaded.values()))
else:
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = "Bearer " + token
    request = urllib.request.Request("https://api.github.com/repos/arpitpaliwal007/agentic-docqa/zipball/main", headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            archive_bytes = response.read()
    except Exception:
        raise RuntimeError("Could not load GitHub source. For this private repository, use Drive ZIP or grant the GITHUB_TOKEN secret repository read access.") from None
    finally:
        headers.clear()
        request = None
        token = None
REPO = Path("/content/agentic-docqa-eval-source") / hashlib.sha256(archive_bytes).hexdigest()[:16]
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    entries = []
    roots = set()
    for entry in archive.infolist():
        path = PurePosixPath(entry.filename)
        if path.is_absolute() or ".." in path.parts or not path.parts or stat.S_ISLNK(entry.external_attr >> 16):
            raise ValueError("Unsafe ZIP entry")
        roots.add(path.parts[0])
        entries.append((entry, path.parts[1:]))
    if len(roots) != 1:
        raise ValueError("Expected one project folder inside the ZIP")
    for entry, parts in entries:
        if not parts or ".git" in parts:
            continue
        target = REPO.joinpath(*parts)
        if target.is_symlink() or any(parent.is_symlink() for parent in target.parents):
            raise ValueError("Unsafe extraction target")
        if entry.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(entry))
if not (REPO / "eval/run_eval.py").is_file():
    raise RuntimeError("This ZIP predates the evaluation harness. Upload or save the updated project ZIP and rerun.")
if "METRIC_POLICY" not in (REPO / "eval/metrics.py").read_text() or "--prefix-ablation" not in (REPO / "eval/run_eval.py").read_text():
    raise RuntimeError("This source predates eval v2. Use the updated agentic-docqa-v2.zip or GitHub main.")
os.chdir(REPO)
print("Project loaded:", REPO)


## 3. Install dependencies and validate fixtures
No extra evaluation packages are required. Install in a subprocess; evaluation also runs in a fresh Python subprocess to avoid stale imports after package changes. GPU-dependent checks remain in the smoke test.

Google Drive also stores package/model download caches so they survive a new runtime.


In [ ]:
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
MODEL_CACHE_ROOT = Path("/content/drive/MyDrive/Agentic DocQA")
RESULTS_DIR = Path("/content/drive/MyDrive/Agentic DocQA/evaluation-results/v2")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.update({"OLLAMA_MODELS":str(MODEL_CACHE_ROOT / "models/ollama"), "HF_HOME":str(MODEL_CACHE_ROOT / "cache/huggingface"), "FASTEMBED_CACHE_PATH":str(MODEL_CACHE_ROOT / "cache/fastembed"), "SENTENCE_TRANSFORMERS_HOME":str(MODEL_CACHE_ROOT / "cache/sentence-transformers"), "PIP_CACHE_DIR":str(MODEL_CACHE_ROOT / "cache/pip")})
subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "eval/run_eval.py", "--validate-only"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


## 4. Install and start Ollama
Use the official installer and keep logs in the v2 results folder. Context is 8192 tokens and one model is loaded at a time to limit memory use. A different judge can cause model reloads between questions, which can affect measured answer latency.


In [ ]:
import json, time
OLLAMA_URL = "http://127.0.0.1:11434"
def ollama_json(path, payload=None, timeout=10):
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(OLLAMA_URL + path, data=data, headers={"Content-Type":"application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.load(response)
def ollama_ready():
    try:
        ollama_json("/api/tags", timeout=2)
        return True
    except Exception:
        return False
if not shutil.which("ollama"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    installer = Path("/content/install-ollama-eval.sh")
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
    subprocess.run(["sh", str(installer)], check=True)
os.environ.update({"OLLAMA_HOST":"127.0.0.1:11434", "OLLAMA_CONTEXT_LENGTH":"8192", "OLLAMA_NUM_PARALLEL":"1", "OLLAMA_MAX_LOADED_MODELS":"1"})
if not ollama_ready():
    with open(str(RESULTS_DIR / "ollama.log"), "a") as log:
        ollama_process = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(), start_new_session=True)
    for _ in range(60):
        if ollama_ready():
            break
        if ollama_process.poll() is not None:
            raise RuntimeError(Path(str(RESULTS_DIR / "ollama.log")).read_text()[-3000:])
        time.sleep(1)
    else:
        raise RuntimeError("Ollama did not respond within 60 seconds. Check the log.")
print("Ollama ready.")


## 5. Choose models and run mode
The judge defaults to the answer model. Set `JUDGE_MODEL` to an installed/supported alternative if desired; it is pulled and GPU-checked before running. A T4's ability to hold `qwen2.5:14b` or `llama3.1:8b` depends on quantization and context; this notebook does not assume either fits completely in VRAM. A failed preflight stops before evaluation. Both judge passes use the selected judge at temperature zero.

`RUN_PREFIX_ABLATION=True` builds separate off/on indexes and measures all six retrieval conditions. Agent answers always use prefixes on. **Rejudge saved answers** reuses original answers and evidence and refreshes every eligible judgment with one v2 policy; it never retains old scores by default.


In [ ]:
ANSWER_MODEL = "qwen2.5:7b" #@param ["qwen2.5:7b", "qwen2.5:3b"]
JUDGE_MODEL = "" #@param {type:"string"}
RUN_PREFIX_ABLATION = True #@param {type:"boolean"}
EVAL_MODE = "Full evaluation" #@param ["Full evaluation", "Rejudge saved answers"]
SAVED_RESULTS = "/content/drive/MyDrive/Agentic DocQA/evaluation-results/rejudged/rejudge-af588f8fd924e7f7.partial.json" #@param {type:"string"}
SELECTED_JUDGE = JUDGE_MODEL.strip() or ANSWER_MODEL
os.environ.update({"EVAL_LLM_MODEL":ANSWER_MODEL, "LLM_MODEL":ANSWER_MODEL, "EVAL_JUDGE_MODEL":SELECTED_JUDGE, "LLM_BASE_URL":OLLAMA_URL + "/v1", "LLM_API_KEY":"ollama", "OLLAMA_HOST":OLLAMA_URL, "NOMIC_PREFIXES":"true"})
models = [ANSWER_MODEL, "nomic-embed-text", SELECTED_JUDGE] if EVAL_MODE == "Full evaluation" else [SELECTED_JUDGE]
for model in dict.fromkeys(models):
    subprocess.run(["ollama", "pull", model], check=True)
def warm_and_verify(model):
    ollama_json("/api/generate", {"model":model,"prompt":"Reply with OK.","stream":False,"keep_alive":"30m","options":{"num_predict":8,"num_ctx":8192}}, timeout=600)
    loaded = ollama_json("/api/ps")["models"]
    item = next((item for item in loaded if item.get("name") in {model, model + ":latest"}), None)
    if not item or item.get("size_vram", 0) == 0:
        raise RuntimeError(f"{model} did not load onto the GPU. Choose a smaller model and rerun this cell.")
    print(model, "GPU bytes:", item.get("size_vram"), "loaded bytes:", item.get("size"))
    if item.get("size_vram", 0) < .9 * item.get("size", 0):
        raise RuntimeError(f"{model} substantially offloaded to CPU. Choose a smaller judge/model for this T4.")
warm_and_verify(SELECTED_JUDGE)
if EVAL_MODE == "Full evaluation" and ANSWER_MODEL != SELECTED_JUDGE:
    warm_and_verify(ANSWER_MODEL)
print("Selected models passed GPU preflight.")


## 6. Mount Drive and choose durable result storage
Completed questions are saved directly here, including raw evidence and judge outputs. The live embedded Qdrant index stays on local runtime disk, isolated from your normal DocQA library. A fresh runtime rebuilds the tiny corpus, then skips compatible completed questions. Package versions and GPU details are part of run identity.


In [ ]:
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")
RESULTS_DIR = Path("/content/drive/MyDrive/Agentic DocQA/evaluation-results/v2")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Durable results:", RESULTS_DIR)


## 7. Smoke test: two questions
Full-evaluation mode executes indexing, six retrieval conditions, the real graph, and two-pass judging on two questions. It writes to `v2/smoke/`. Rejudge-only mode skips this step because it reuses the saved pipeline outputs.


In [ ]:
def run_eval_logged(command, log_name):
    with open(RESULTS_DIR / log_name, "a", encoding="utf-8") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
PREFIX_FLAG = "--prefix-ablation" if RUN_PREFIX_ABLATION else "--no-prefix-ablation"
if EVAL_MODE == "Full evaluation":
    run_eval_logged([sys.executable,"-u","eval/run_eval.py","--model",ANSWER_MODEL,"--judge-model",SELECTED_JUDGE,PREFIX_FLAG,"--limit","2","--out",str(RESULTS_DIR / "smoke")], "smoke-console.log")
else:
    print("Rejudge-only mode: smoke pipeline skipped.")


## 8. Full evaluation or uniform rejudging (resumable)
Full evaluation rebuilds only the isolated eval user and both prefix-condition indexes, then runs all 20 questions. Each completed question is saved to Drive. Repeat the same command to resume compatible results.

Rejudge mode refreshes ALL saved eligible answers with the same current policy and both judge passes. It writes to `v2/rejudged-all/`, preserves original answer/retrieval provenance, and does not replace any original result file. Neither mode uses selective retention of old scores.


In [ ]:
if EVAL_MODE == "Full evaluation":
    REPORT_DIR = RESULTS_DIR
    run_eval_logged([sys.executable,"-u","eval/run_eval.py","--model",ANSWER_MODEL,"--judge-model",SELECTED_JUDGE,PREFIX_FLAG,"--out",str(REPORT_DIR)], "full-console.log")
else:
    REPORT_DIR = RESULTS_DIR / "rejudged-all"
    if not Path(SAVED_RESULTS).is_file():
        raise FileNotFoundError("Set SAVED_RESULTS to the original JSON/partial JSON report in Drive.")
    run_eval_logged([sys.executable,"-u","eval/rejudge.py",SAVED_RESULTS,"--judge-model",SELECTED_JUDGE,"--out",str(REPORT_DIR)], "rejudge-all-console.log")


## 9. Display the report
The report contains the two-pass judge mean and disagreement rate, independent key-fact recall, refusal diagnostics, type breakdowns, prefix comparison, agent retrieval and actual queries in failure examples. Missing or unrun observations remain N/A. No benchmark numbers are prefilled.


In [ ]:
from IPython.display import Markdown, display
reports = sorted(REPORT_DIR.glob("*.md"), key=lambda path:path.stat().st_mtime)
if not reports:
    raise RuntimeError("No report exists yet. Run the smoke/full or rejudge cell first.")
report = reports[-1]
print("Report:", report)
display(Markdown(report.read_text()))
